[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.6_custom_silicon/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.6_custom_silicon/lab.ipynb)

# Lab 8.6: Custom Silicon for LLM Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.6_custom_silicon/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.ai/open?repo=harshuljain13/llm-inference-at-scale&path=content/09_operations/08.6_custom_silicon/lab.ipynb&branch=master)

This lab models the decode throughput arithmetic for custom silicon vs GPUs, computes the flexibility-efficiency tradeoff, and visualizes break-even economics.

In [ ]:
# Install dependencies via subprocess (avoids kernel restart)
import subprocess
import sys
# Only numpy and matplotlib needed for simulation
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

In [ ]:
import numpy as np  # numerical computation
import matplotlib.pyplot as plt  # visualization
from dataclasses import dataclass  # structured chip specifications


@dataclass
class ChipSpec:
    """Specification for an inference accelerator chip."""
    name: str  # human-readable chip name
    bandwidth_tb_s: float  # memory bandwidth in TB/s
    compute_tflops: float  # peak FP16 compute in TFLOPS
    on_chip_sram_gb: float  # on-chip SRAM capacity in GB
    cost_per_hour: float  # cloud/rental cost per hour in USD
    color: str  # plot color for visualization


# Define the chip landscape (real specs from public sources)
CHIPS = [
    ChipSpec('H100 (GPU)', 3.35, 989, 0.05, 30.0, '#ef4444'),  # NVIDIA flagship
    ChipSpec('Groq LPU', 80.0, 200, 0.23, 15.0, '#3b82f6'),  # SRAM-only, deterministic
    ChipSpec('Cerebras WSE-3', 21000.0, 500, 44.0, 50.0, '#8b5cf6'),  # wafer-scale
    ChipSpec('Inferentia2', 2.4, 380, 0.032, 12.98, '#10b981'),  # AWS cost-optimized
    ChipSpec('A100 (GPU)', 2.0, 312, 0.04, 15.0, '#f97316'),  # previous gen baseline
]

# Model configuration: Llama 70B in FP16
MODEL_SIZE_GB = 140  # 70B params * 2 bytes (FP16)
# This is the memory that must be read for EVERY token generated
MODEL_NAME = 'Llama 70B (FP16)'

print(f"Model: {MODEL_NAME} ({MODEL_SIZE_GB} GB)")
print(f"Chips: {len(CHIPS)} architectures compared")

In [ ]:
# --- Decode Throughput Arithmetic ---
# Fundamental formula: decode tok/s = bandwidth / model_size
# This is the HARD LIMIT: each token requires reading all parameters once

def compute_decode_throughput(chip: ChipSpec, model_gb: float, efficiency: float = 0.7) -> float:
    """Compute theoretical decode tokens/second for a chip.
    
    Efficiency accounts for routing overhead, synchronization, and
    imperfect memory access patterns in real transformers.
    """
    # Convert TB/s to GB/s for consistent units
    bandwidth_gb_s = chip.bandwidth_tb_s * 1000
    # Tokens per second = effective bandwidth / bytes per token generation
    theoretical_tok_s = bandwidth_gb_s / model_gb
    # Apply real-world efficiency factor
    practical_tok_s = theoretical_tok_s * efficiency
    return practical_tok_s


# Compute decode throughput for each chip
print(f"{'Chip':<20} {'Bandwidth':<15} {'Theoretical':<15} {'Practical (70%)':<15} {'Speedup vs H100':<15}")
print("-" * 80)

# Store results for plotting
chip_names = []  # names for bar chart x-axis
chip_throughputs = []  # practical decode tok/s
chip_colors = []  # colors per chip

# H100 baseline for speedup calculation
h100_throughput = compute_decode_throughput(CHIPS[0], MODEL_SIZE_GB)
# H100 is the baseline all custom silicon is measured against

for chip in CHIPS:
    # Compute both theoretical and practical throughput
    theoretical = compute_decode_throughput(chip, MODEL_SIZE_GB, efficiency=1.0)
    practical = compute_decode_throughput(chip, MODEL_SIZE_GB, efficiency=0.7)
    # Speedup relative to H100
    speedup = practical / h100_throughput
    # Print comparison table
    print(f"{chip.name:<20} {chip.bandwidth_tb_s:<15.1f}TB/s {theoretical:<15.0f}tok/s {practical:<15.0f}tok/s {speedup:<15.1f}x")
    # Accumulate for plotting
    chip_names.append(chip.name)
    chip_throughputs.append(practical)
    chip_colors.append(chip.color)

In [ ]:
# --- Visualization: Decode Throughput Comparison ---
fig_c3, ax_c3 = plt.subplots(1, 1, figsize=(10, 6))

# Bar chart of decode throughput (log scale needed due to 6000x range)
bars = ax_c3.bar(chip_names, chip_throughputs, color=chip_colors, edgecolor='black', linewidth=0.8)
ax_c3.set_yscale('log')  # log scale to show the massive differences
ax_c3.set_ylabel('Decode Tokens/sec (log scale)', fontsize=12)
ax_c3.set_title(f'Decode Throughput: {MODEL_NAME}\n(bandwidth / model_size, 70% efficiency)', fontsize=13)
ax_c3.grid(True, axis='y', alpha=0.3)

# Annotate each bar with its value
for bar_item, throughput in zip(bars, chip_throughputs):
    # Place text above each bar
    ax_c3.text(bar_item.get_x() + bar_item.get_width()/2, throughput * 1.3,
            f'{throughput:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- Cost per Million Tokens Analysis ---
# Real economics: cost depends on THROUGHPUT, not just chip price

def compute_cost_per_million_tokens(chip: ChipSpec, model_gb: float) -> float:
    """Calculate cost per 1M output tokens for each chip.
    
    Cost/token = hourly_rate / (tok/s * 3600 seconds/hour)
    """
    # Get practical throughput
    tok_per_sec = compute_decode_throughput(chip, model_gb)
    # Tokens generated per hour
    tok_per_hour = tok_per_sec * 3600
    # Cost per token
    cost_per_token = chip.cost_per_hour / tok_per_hour
    # Cost per million tokens
    cost_per_million = cost_per_token * 1_000_000
    return cost_per_million


# Compute and display cost comparison
costs = []  # store for plotting
print(f"{'Chip':<20} {'Tok/s':<12} {'$/M tokens':<12} {'Savings vs H100':<15}")
print("-" * 60)

# H100 baseline cost
h100_cost = compute_cost_per_million_tokens(CHIPS[0], MODEL_SIZE_GB)

for chip in CHIPS:
    # Cost per million output tokens
    cost = compute_cost_per_million_tokens(chip, MODEL_SIZE_GB)
    # Savings percentage relative to H100
    savings_pct = (1 - cost / h100_cost) * 100
    tok_s = compute_decode_throughput(chip, MODEL_SIZE_GB)
    print(f"{chip.name:<20} {tok_s:<12.0f} ${cost:<11.4f} {savings_pct:+.0f}%")
    costs.append(cost)

# Visualize cost comparison
fig_c4, ax_c4 = plt.subplots(1, 1, figsize=(10, 5))
bars = ax_c4.bar(chip_names, costs, color=chip_colors, edgecolor='black', linewidth=0.8)
ax_c4.set_ylabel('Cost per Million Tokens ($)', fontsize=12)
ax_c4.set_title('Inference Cost Comparison (output tokens, decode phase)', fontsize=13)
ax_c4.grid(True, axis='y', alpha=0.3)

# Annotate bars with dollar values
for bar_item, cost in zip(bars, costs):
    ax_c4.text(bar_item.get_x() + bar_item.get_width()/2, cost + max(costs)*0.02,
            f'${cost:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# --- Utilization Sensitivity Analysis ---
# Custom silicon economics are EXTREMELY sensitive to utilization
# At low utilization, GPU flexibility wins (serve other models during idle time)

# Sweep utilization from 10% to 100%
utilization_range = np.arange(0.1, 1.05, 0.05)
# Utilization is the critical factor in custom silicon economics

def effective_cost_at_utilization(chip: ChipSpec, model_gb: float, utilization: float) -> float:
    """Cost per M tokens adjusted for utilization.
    
    At low utilization, you pay full hourly rate but generate fewer tokens.
    GPU can fill idle time with other workloads; custom silicon cannot.
    """
    # Base cost assumes 100% utilization
    base_cost = compute_cost_per_million_tokens(chip, model_gb)
    # At lower utilization, effective cost increases (paying for idle time)
    effective_cost = base_cost / utilization
    return effective_cost


# Plot utilization curves for key chips
fig_c5, ax_c5 = plt.subplots(1, 1, figsize=(10, 6))

# Only plot the most relevant chips for clarity
key_chips = [CHIPS[0], CHIPS[1], CHIPS[2], CHIPS[3]]  # H100, Groq, Cerebras, Inferentia

for chip in key_chips:
    # Compute cost across utilization range
    costs_at_util = [effective_cost_at_utilization(chip, MODEL_SIZE_GB, u) for u in utilization_range]
    ax_c5.plot(utilization_range * 100, costs_at_util,
            linewidth=2, label=chip.name, color=chip.color)

# Mark the crossover zones
ax_c5.axhline(y=h100_cost, color='gray', linestyle=':', alpha=0.5, label='H100 @ 100% baseline')
ax_c5.set_xlabel('GPU/Chip Utilization (%)', fontsize=12)
ax_c5.set_ylabel('Effective Cost per M Tokens ($)', fontsize=12)
ax_c5.set_title('Custom Silicon Economics by Utilization\n(low util favors GPU flexibility)', fontsize=13)
ax_c5.legend(fontsize=10)
ax_c5.grid(True, alpha=0.3)
ax_c5.set_ylim(0, h100_cost * 5)  # cap y-axis for readability
plt.tight_layout()
plt.show()

# Print break-even utilization for each custom chip vs H100 at 90%
print("\nBreak-even utilization (vs H100 at 90% util):")
h100_effective = effective_cost_at_utilization(CHIPS[0], MODEL_SIZE_GB, 0.9)
for chip in key_chips[1:]:  # skip H100 itself
    # Find utilization where custom silicon matches H100 at 90%
    base = compute_cost_per_million_tokens(chip, MODEL_SIZE_GB)
    # breakeven: base/util = h100_effective => util = base/h100_effective
    breakeven_util = base / h100_effective
# Below this utilization, GPU wins due to multi-workload flexibility
    if breakeven_util <= 1.0:
        print(f"  {chip.name}: breaks even at {breakeven_util*100:.0f}% utilization")
    else:
        print(f"  {chip.name}: always more expensive than H100 at 90%")

In [ ]:
# --- Flexibility-Efficiency Spectrum Visualization ---
# Each chip trades flexibility for efficiency in a different way

# Assign flexibility scores (1-10 scale, manually calibrated from ecosystem maturity)
# Higher = more flexible (supports more models, faster deployment, multi-workload)
flexibility_scores = {
# Scores based on: model support breadth, SDK maturity, deployment speed
    'H100 (GPU)': 9.5,  # universal: any model, any workload, CUDA ecosystem
    'Groq LPU': 3.0,  # inference-only, limited models, requires compilation
    'Cerebras WSE-3': 4.0,  # training+inference but limited SDK, few models
    'Inferentia2': 6.0,  # good Neuron SDK, major models supported, AWS-only
    'A100 (GPU)': 9.0,  # same CUDA ecosystem, slightly less compute
}

# Efficiency = tokens per dollar (inverse of cost per M tokens)
efficiency_scores = {chip.name: 1.0 / compute_cost_per_million_tokens(chip, MODEL_SIZE_GB)
# Derived directly from cost modeling above
                     for chip in CHIPS}

# Scatter plot: flexibility vs efficiency
fig_c6, ax_c6 = plt.subplots(1, 1, figsize=(10, 7))

for chip in CHIPS:
    flex = flexibility_scores[chip.name]  # x-axis: flexibility
    eff = efficiency_scores[chip.name]  # y-axis: efficiency (tok/$)
    # Plot each chip as a labeled point
    ax_c6.scatter(flex, eff, s=200, c=chip.color, edgecolors='black', linewidth=1.5, zorder=5)
    # Label offset to avoid overlap
    ax_c6.annotate(chip.name, (flex, eff), textcoords='offset points',
                xytext=(10, 10), fontsize=11, fontweight='bold')

# Draw the tradeoff frontier (conceptual)
ax_c6.set_xlabel('Flexibility (ecosystem maturity, model support)', fontsize=12)
ax_c6.set_ylabel('Efficiency (tokens per dollar)', fontsize=12)
ax_c6.set_title('The Flexibility-Efficiency Tradeoff in Inference Hardware', fontsize=14)
ax_c6.grid(True, alpha=0.3)

# Add quadrant labels
ax_c6.text(8.5, max(efficiency_scores.values()) * 0.9, 'Ideal\n(high flex + high eff)',
        fontsize=9, color='green', ha='center', style='italic')
ax_c6.text(3.5, min(efficiency_scores.values()) * 1.5, 'Specialized\nbut expensive',
        fontsize=9, color='red', ha='center', style='italic')

plt.tight_layout()
plt.show()

print("\nKey insight: No chip occupies the top-right (high flexibility + high efficiency).")
print("The tradeoff is fundamental: specialization buys efficiency at the cost of flexibility.")